## Initialization

In [ ]:
# Importing needed code

import re
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    TypeVar,
    Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.stats import linregress

from data_processing.arc_paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    find_failed_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.helpers import (
    stop, get_input_with_default, input_experiment_ids, get_midpoints_from_min_max_series)
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
def load_rate_data(experiment_name: str, time_bin_length: int) -> pd.DataFrame:
    file_name = f"{experiment_name}_data_{time_bin_length}s_bin.csv"
    output_root = get_report_root(experiment_name)
    file_path = output_root / file_name
    df = pd.read_csv(file_path)
    return df

In [ ]:
T = TypeVar("T")


def input_with_validation(
    converter: Callable[[Any], T],
    prompt: str,
    invalid_msg: str
) -> T:
    output = None
    while output is None:
        in_val = input(prompt)
        try:
            output = converter(in_val)
        except ValueError:
            print(invalid_msg)
    return output


def input_yes_no(
    prompt: str,
    default_yes: bool = True
) -> bool:
    possible_yes = ['y', 'yes']
    possible_no = ['n', 'no']
    full_prompt = f"{prompt} (y/n)\nPress Enter for {'yes' if default_yes else 'no'}"

    in_val = input(full_prompt)
    if in_val.lower() in possible_yes:
        return True
    elif in_val.lower() in possible_no:
        return False
    else:
        return default_yes

In [ ]:
TimePeriod = tuple[float, float]  # in ps after exp. start
TimePeriods = list[TimePeriod]


def get_time_period_borders(
    rate_df: pd.DataFrame,
    duration: float | None = None
) -> TimePeriod:
    # TODO let user enter stable regions
    # check stability via linear fit
    # show slope to user
    # let user confirm stable region, or re-enter region
    done = False
    # start = None
    while not done:
        end = None
        start = input_with_validation(
            float,
            "Enter start time (in minutes after experiment start): ",
            "Not a valid number, try again"
        )
        # while start is None:
        #     in_val = input("Enter start time (in minutes after experiment start): ")
        #     try:
        #         start = float(in_val)
        #     except ValueError:
        #         print("Not a valid number, try again")
        if duration is not None:
            end = start + duration
        if end is None:
            end = input_with_validation(
                float,
                "Enter end time (in minutes after experiment start): ",
                "Not a valid number, try again"
            )
            # in_val = input("Enter start time (in minutes after experiment start): ")
            # try:
            #     end = float(in_val)
            # except ValueError:
            #     print("Not a valid number, try again")
        if end <= start:
            print("End must be greater than start, try again")
            # start = None
            # end = None
            continue

        bin_time = rate_df["Bin time (s)"]
        if end > bin_time.max():
            end = bin_time.max()
#         y_all = rate_df["Neutron rate (cps)"]
#         duration_series_idx = x_all.between(start * 60, end * 60)
#         x = x_all[duration_series_idx]
#         y = y_all[duration_series_idx]
#         result = linregress(x, y)
#         print(f"Rate slope = {result.slope:.2E} cps/s +/- {result.stderr:.2E}")
#         in_done = get_input_with_default(
#             """\
# Is this region stable?
# Enter y/n, or press Enter for no
# """,
#             "n",
#             str
#         )
#         done = in_done != "n"
    return start, end


def get_time_periods(
    rate_df: pd.DataFrame,
    count: int = 3,
    persist_duration: bool = False,
) -> TimePeriods:
    if count < 1:
        raise ValueError("count must be 1 or greater")

    regions = []

    print("---Region 1---")
    bg_region = get_time_period_borders(rate_df)
    regions.append(bg_region)
    bg_duration = None
    if persist_duration:
        bg_start, bg_end = bg_region
        bg_duration = bg_end - bg_start

    for i in range(count - 1):
        print(f"---Region {i+2}---")
        region = get_time_period_borders(rate_df, bg_duration)
        regions.append(region)

    return regions

In [ ]:
# start with dataframe with neutron rates
# scan through dataframe from start time to end time
# for each row (use normal iteration):
# - find rows 30 mins ahead
# - measure stability, return True or False


def get_rate_stability_series(
    rate_df: pd.DataFrame,
    stability_threshold: float = 5,  # percent variance allowed
    stability_period: float = 30,  # stable over this time period
    search_period: TimePeriod | None = None
) -> pd.Series:
    stable_thresh = stability_threshold / 100
    time_series = rate_df["Bin time (s)"]

    if search_period is None:
        start_time = time_series.min()
        end_time = time_series.max()
    else:
        start_time, end_time = search_period
        start_time = start_time * 60
        end_time = end_time + stability_period
        end_time = end_time * 60 if end_time <= time_series.max() else time_series.max()

    search_rate_df = rate_df[time_series.between(start_time, end_time)]
    stable_start_data = []
    for index, iter_start_time in time_series.items():
        # print(index)
        # print(row)
        # iter_start_time = row['Bin time (s)']
        iter_end_time = iter_start_time + (stability_period*60)
        iter_df = rate_df[time_series.between(iter_start_time, iter_end_time)]
        neutron_rates = iter_df['Neutron rate (cps)']
        mean_rate = neutron_rates.mean()
        max_rate = neutron_rates.max()
        min_rate = neutron_rates.min()
        max_variance = abs(max_rate - mean_rate)
        min_variance = abs(min_rate - mean_rate)
        max_percent_variance = max_variance / mean_rate
        min_percent_variance = min_variance / mean_rate
        max_stable = max_percent_variance <= stable_thresh
        min_stable = min_percent_variance <= stable_thresh
        is_stable = max_stable and min_stable
        if is_stable:
            print(f"{iter_start_time/60:.2f}: {mean_rate:.2f},{min_rate:.2f},{max_rate:.2f}")
        stable_start_data.append(is_stable)
    stable_start_series = pd.Series(stable_start_data, search_rate_df.index)
    stability_data = []
    for index, time_value in time_series.items():
        start_search = time_value - (stability_period*60)
        search_data = stable_start_series[time_series.between(start_search, time_value)]
        is_in_stable_period = search_data.any()
        stability_data.append(is_in_stable_period)
    stability_series = pd.Series(stability_data, search_rate_df.index)
    return stability_series

## Input Settings

In [ ]:
print("Enter E-cell experiment IDs")
ecell_experiment_ids = input_experiment_ids()
print()
print("Enter beam-loading experiment IDs")
beam_experiment_ids = input_experiment_ids()
all_experiment_ids = list(zip(beam_experiment_ids, ecell_experiment_ids))

In [ ]:
time_bin_length = get_input_with_default(
    """\
Enter bin length (in seconds)
Press Enter for default (30)
""",
    30,
    int
)

## Data Loading

In [ ]:
LinkableDataset = dict[str, dict[ExperimentDataKey, Any] | str]
LinkedDatasets = tuple[LinkableDataset, LinkableDataset]
all_experiment_data: list[LinkedDatasets] = [
    (
        {"exp_id": beam_exp_id, "data": {}},
        {"exp_id": ecell_exp_id, "data": {}}
    )
    for beam_exp_id, ecell_exp_id in all_experiment_ids
]

In [ ]:
for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']
    for exp_id, exp_data in [(beam_id, beam_data), (ecell_id, ecell_data)]:
        detector_df = load_psd(exp_id)
        rate_df = load_rate_data(exp_id, time_bin_length)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = detector_df
        exp_data[ExperimentDataKey.ALL_BINNED_DATA] = rate_df

In [ ]:
# print(all_experiment_data)

In [ ]:
experiment_beam = all_experiment_data[0][0]
experiment_ecell = all_experiment_data[0][1]
beam_exp_id = experiment_beam['exp_id']
ecell_exp_id = experiment_ecell['exp_id']
print(beam_exp_id)
print(ecell_exp_id)
beam_exp_data = experiment_beam['data']
ecell_exp_data = experiment_ecell['data']
beam_exp_df = beam_exp_data[ExperimentDataKey.ALL_BINNED_DATA]
ecell_exp_df = ecell_exp_data[ExperimentDataKey.ALL_BINNED_DATA]
# exp_df.head()

In [ ]:
beam_event_rates = beam_exp_df['Neutron rate (cps)'] + beam_exp_df['Background gamma rate (cps)']
ecell_event_rates = ecell_exp_df['Neutron rate (cps)'] + ecell_exp_df['Background gamma rate (cps)']
beam_bin_time = beam_exp_df['Bin time (s)']
ecell_bin_time = ecell_exp_df['Bin time (s)']
beam_bin_minutes = beam_bin_time / 60
ecell_bin_minutes = ecell_bin_time / 60

fig, ax = plt.subplots(figsize=(10, 8))
dot_size = 4

ax.plot(
    beam_bin_minutes,
    beam_event_rates,
    ".",
    linestyle='-',
    # markersize=dot_size,
    # alpha=0.2,
    label="Beam loading"
)
ax.plot(
    ecell_bin_minutes,
    ecell_event_rates,
    ".",
    linestyle='-',
    # markersize=dot_size,
    # alpha=0.2,
    label="Beam+Ecell"
)
ax.grid(which="both")
ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
ax.set_ylabel("Particle count rate [1/s]", fontsize=14)
# ax.set_ylim(0, 22)
# ax.set_ylim(3.5, 5.0)
ax.tick_params(labelsize=12)
ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

# Add a title above the plot
fig.text(
    0.5,
    0.90,
    f"Particle count rate over time (Dwell time {time_bin_length}s)",
    ha='center',
    fontsize=20
)
ax.legend()

# Show the plot (optional)
plt.show()

In [ ]:
beam_background_rates = beam_event_rates[beam_event_rates <= 125]
beam_early_bg = beam_background_rates[beam_bin_minutes < 30]
beam_late_bg = beam_background_rates[beam_bin_minutes > 90]

ecell_early_bg_times = beam_bin_minutes[beam_early_bg.index]
bg_interval = ecell_early_bg_times.iloc[1] - ecell_early_bg_times.iloc[0]
beam_early_bg_start = ecell_early_bg_times.min() - bg_interval/2
beam_early_bg_end = ecell_early_bg_times.max() + bg_interval/2

ecell_late_bg_times = beam_bin_minutes[beam_late_bg.index]
bg_interval = ecell_late_bg_times.iloc[1] - ecell_late_bg_times.iloc[0]
beam_late_bg_start = ecell_late_bg_times.min() - bg_interval/2
beam_late_bg_end = ecell_late_bg_times.max() + bg_interval/2

beam_mean_bg_rate = beam_background_rates.mean()
beam_adjusted_rates = beam_event_rates - beam_mean_bg_rate

In [ ]:
ecell_background_rates = ecell_event_rates[ecell_event_rates <= 125]
ecell_early_bg = ecell_background_rates[ecell_bin_minutes < 30]
ecell_late_bg = ecell_background_rates[ecell_bin_minutes > 90]

ecell_early_bg_times = ecell_bin_minutes[ecell_early_bg.index]
bg_interval = ecell_early_bg_times.iloc[1] - ecell_early_bg_times.iloc[0]
ecell_early_bg_start = ecell_early_bg_times.min() - bg_interval/2
ecell_early_bg_end = ecell_early_bg_times.max() + bg_interval/2

ecell_late_bg_times = ecell_bin_minutes[ecell_late_bg.index]
bg_interval = ecell_late_bg_times.iloc[1] - ecell_late_bg_times.iloc[0]
ecell_late_bg_start = ecell_late_bg_times.min() - bg_interval/2
ecell_late_bg_end = ecell_late_bg_times.max() + bg_interval/2

ecell_mean_bg_rate = ecell_background_rates.mean()
# print(background_rates)
print(ecell_mean_bg_rate)
ecell_adjusted_rates = ecell_event_rates - ecell_mean_bg_rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
dot_size = 4

# bin_minutes = bin_time / 60
# ax.plot(
#     bin_minutes,
#     adjusted_rates,
#     ".",
#     linestyle='-',
#     # markersize=dot_size,
#     # alpha=0.2,
#     # label=label
# )
ax.plot(
    beam_bin_minutes,
    beam_adjusted_rates,
    ".",
    linestyle='-',
    # markersize=dot_size,
    # alpha=0.2,
    label="Beam loading"
)
ax.plot(
    ecell_bin_minutes,
    ecell_adjusted_rates,
    ".",
    linestyle='-',
    # markersize=dot_size,
    # alpha=0.2,
    label="Beam+Ecell"
)
ax.grid(which="both")
ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
ax.set_ylabel("Particle count rate [1/s]", fontsize=14)
# ax.set_ylim(0, 22)
# ax.set_ylim(3.5, 5.0)
ax.tick_params(labelsize=12)
ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

# Add a title above the plot
fig.text(
    0.5,
    0.90,
    f"Particle count rate over time (Dwell time {time_bin_length}s)",
    ha='center',
    fontsize=20
)
ax.legend()

# Show the plot (optional)
plt.show()

In [ ]:
threshold = 5

print("Beam")
beam_stability = get_rate_stability_series(beam_exp_df, stability_threshold=threshold)
beam_stable = beam_exp_df[beam_stability]

print("Ecell")
ecell_stability = get_rate_stability_series(ecell_exp_df, stability_threshold=threshold)
ecell_stable = ecell_exp_df[ecell_stability]

# beam_stable

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
dot_size = 8

beam_stable_bin_minutes = beam_bin_minutes[beam_stable.index]
beam_stable_adjusted_rates = beam_adjusted_rates[beam_stable.index]
ecell_stable_bin_minutes = ecell_bin_minutes[ecell_stable.index]
ecell_stable_adjusted_rates = ecell_adjusted_rates[ecell_stable.index]


# bin_minutes = bin_time / 60
# ax.plot(
#     bin_minutes,
#     adjusted_rates,
#     ".",
#     linestyle='-',
#     # markersize=dot_size,
#     # alpha=0.2,
#     # label=label
# )
ax.plot(
    beam_bin_minutes,
    beam_adjusted_rates,
    "",
    # linestyle='-',
    # markersize=dot_size,
    # alpha=0.2,
    label="Beam loading"
)
ax.plot(
    ecell_bin_minutes,
    ecell_adjusted_rates,
    "",
    # linestyle='-',
    # markersize=dot_size,
    # alpha=0.2,
    label="Beam+Ecell"
)
ax.plot(
    beam_stable_bin_minutes,
    beam_stable_adjusted_rates,
    ".",
    linestyle='',
    markersize=dot_size,
    alpha=0.5,
    label="Stable (Beam)"
)
ax.plot(
    ecell_stable_bin_minutes,
    ecell_stable_adjusted_rates,
    "x",
    linestyle='',
    markersize=dot_size,
    alpha=0.5,
    label="Stable (Ecell)"
)
ax.grid(which="both")
ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
ax.set_ylabel("Particle count rate [1/s]", fontsize=14)
# ax.set_ylim(0, 22)
# ax.set_ylim(3.5, 5.0)
ax.tick_params(labelsize=12)
ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

# Add a title above the plot
fig.text(
    0.5,
    0.90,
    f"Particle count rate over time (Dwell time {time_bin_length}s)",
    ha='center',
    fontsize=20
)
ax.legend()

# Show the plot (optional)
plt.show()

In [ ]:
test_df = pd.DataFrame(dict(values=list(range(100))))
test_series = test_df['values']
test_series = test_series[test_series.between(25, 50)]
test_series_min = test_series.min()
test_series_max = test_series.max()
test_series_data = list(range(test_series_min, test_series_max+1))
test_series_data = [x*2 for x in test_series_data]
new_test_series = pd.Series(test_series_data, test_series.index)
print(new_test_series)
# test_series2 = test_df['values']
# test_series2 = test_series2[test_series2.between(45, 65)]
# test_series2 = test_series2 * 3
# combined_series = test_series.combine_first(test_series2)
# print(combined_series)
# test_df['modified'] = combined_series
# test_df['test'] = test_series
# test_df.loc[20:65]

## Analysis

In [ ]:
bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value
dot_size = 8

for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    beam_label = f"Beam ({beam_id})"
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']
    ecell_label = f"Ecell ({ecell_id})"

    beam_neutrons = beam_data[ExperimentDataKey.ALL_BINNED_DATA]
    ecell_neutrons = ecell_data[ExperimentDataKey.ALL_BINNED_DATA]

    fig, ax = plt.subplots(figsize=(10, 8))

    for label, binned_neutrons in [
        (beam_label, beam_neutrons),
        (ecell_label, ecell_neutrons)
    ]:
        zeroed_bins = binned_neutrons[bin_time_col_name] / 60
        rates = binned_neutrons[n_rate_col_name]
        rate_errors = binned_neutrons[n_error_col_name]
        ax.errorbar(
            zeroed_bins,
            rates,
            yerr=rate_errors,
            fmt=".",
            linestyle='',
            markersize=dot_size,
            capsize=dot_size,
            alpha=0.2,
            label=label            
        )
    ax.grid(which="both")
    ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("Neutron count rate [1/s]", fontsize=14)
    # ax.set_ylim(0, 22)
    # ax.set_ylim(3.5, 5.0)
    ax.tick_params(labelsize=12)
    ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

    # Add a title above the plot
    fig.text(
        0.5,
        0.90,
        f"Neutron count rate over time (Dwell time {time_bin_length}s)",
        ha='center',
        fontsize=20
    )
    fig.legend()

    # Show the plot (optional)
    plt.show()

In [ ]:
# let user enter stable regions
# check stability via linear fit
# show slope to user
# let user confirm stable region, or re-enter region

for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']

    print(f"-----{beam_id}/{ecell_id}-----")
    persist_duration = input_yes_no(
        "Keep the same duration for all regions?", default_yes=False
    )
    final_background = input_yes_no(
        "Use a final background region?"
    )
    region_count = 4 if final_background else 3

    # event_counts = {"Beam": [], "Ecell": []}
    # durations = {"Beam": [], "Ecell": []}

    for exp_type, exp_id, exp_data in [
        ("Ecell", ecell_id, ecell_data),
        ("Beam", beam_id, beam_data)
    ]:
        adj_region_count = region_count if exp_type == "Ecell"\
            else region_count - 2
        if adj_region_count == 4:
            region_type_msg = ("first background, beam-loading, ecell, " +
                               "final background")
        elif adj_region_count == 3:
            region_type_msg = "background, beam-loading, ecell"
        elif adj_region_count == 2:
            region_type_msg = "first background, final background"
        elif adj_region_count == 1:
            region_type_msg = "background"
        else:
            region_type_msg = "UNKNOWN COUNT"

        event_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
        rate_df = exp_data[ExperimentDataKey.ALL_BINNED_DATA]

        print(f"---{exp_type}: {exp_id}---")
        print(f"Select stable regions: {region_type_msg}")
        stable_regions = find_stable_regions(
            rate_df,
            count=adj_region_count,
            persist_duration=persist_duration,
        )
        print()

        first_background, *remaining_regions = stable_regions
        stable_regions_dict = {'first_bg': first_background}
        if len(remaining_regions) > 1:
            beam_region, ecell_region, *remaining_regions = remaining_regions
            stable_regions_dict['beam'] = beam_region
            stable_regions_dict['ecell'] = ecell_region
        if len(remaining_regions) > 0:
            stable_regions_dict['final_bg'] = remaining_regions[0]
        exp_data['stable_regions'] = stable_regions_dict
    print()

In [ ]:
# display results
for linked_dataset in all_experiment_data:
    beam_dataset, ecell_dataset = linked_dataset
    beam_id = beam_dataset['exp_id']
    beam_data = beam_dataset['data']
    beam_regions = beam_data['stable_regions']
    ecell_id = ecell_dataset['exp_id']
    ecell_data = ecell_dataset['data']
    ecell_regions = ecell_data['stable_regions']
    if 'beam' not in beam_regions and 'beam' in ecell_regions:
        beam_regions['beam'] = ecell_regions['beam']
    if 'ecell' not in beam_regions and 'ecell' in ecell_regions:
        beam_regions['ecell'] = ecell_regions['ecell']

    neutron_rate_data = {"Beam": {}, "Ecell": {}}
    event_counts = {"Beam": {}, "Ecell": {}}
    durations = {"Beam": {}, "Ecell": {}}

    for exp_type, exp_id, exp_data, stable_regions in [
        ("Ecell", ecell_id, ecell_data, ecell_regions),
        ("Beam", beam_id, beam_data, beam_regions)
    ]:
        event_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
        rate_df = exp_data[ExperimentDataKey.ALL_BINNED_DATA]

        print(f"-----Experiment {exp_id}-----")
        print("---Stable Regions---")
        for i, region_item in enumerate(stable_regions.items()):
            key, region = region_item
            start, end = region
            duration = end - start
            print(f"Region {i+1}: {start:.1f} min to {end:.1f} min")
            print(f"Duration: {duration:.1f} min")

            time_col = event_df['TIMETAG']
            # count = event_df[event_df[
            #     (start * 1e12) < time_col & time_col <= (end * 1e12)
            # ]].shape[0]
            is_in_region = time_col.between(start * 60 * 1e12, end * 60 * 1e12)
            events_in_region = event_df[
                is_in_region
            ]
            event_count = events_in_region.shape[0]
            durations[exp_type][key] = duration
            event_counts[exp_type][key] = event_count

            if 'bg' in key:
                rate_time_col = rate_df['Bin time (s)']
                rate_is_in_region = rate_time_col.between(start * 60, end * 60)
                rate_in_region = rate_df[rate_is_in_region]
                neutron_rate_data[exp_type][key] = rate_in_region

    beam_counts = event_counts["Beam"]
    beam_durations = durations["Beam"]
    beam_n_rates = neutron_rate_data["Beam"]
    ecell_counts = event_counts["Ecell"]
    ecell_durations = durations["Ecell"]
    ecell_n_rates = neutron_rate_data["Ecell"]
    # TODO get bg neutron rates, combine, find mean

    beam_bg_count = beam_counts['first_bg']
    if 'final_bg' in beam_counts:
        beam_bg_count += beam_counts['final_bg']
    beam_beam_count = beam_counts['beam']
    beam_ecell_count = beam_counts['ecell']
    beam_bg_duration = beam_durations['first_bg']
    if 'final_bg' in beam_durations:
        beam_bg_duration += beam_durations['final_bg']
    beam_beam_duration = beam_durations['beam']
    beam_ecell_duration = beam_durations['ecell']
    if 'final_bg' in beam_n_rates:
        beam_n_bg_df = pd.concat([beam_n_rates['first_bg'],
                                  beam_n_rates['final_bg']])
    else:
        beam_n_bg_df = beam_n_rates['first_bg']
    beam_n_bg_rate = beam_n_bg_df['Neutron rate (cps)'].mean()

    ecell_bg_count = ecell_counts['first_bg']
    if 'final_bg' in ecell_counts:
        ecell_bg_count += ecell_counts['final_bg']
    ecell_beam_count = ecell_counts['beam']
    ecell_ecell_count = ecell_counts['ecell']
    ecell_bg_duration = ecell_durations['first_bg']
    if 'final_bg' in ecell_durations:
        ecell_bg_duration += ecell_durations['final_bg']
    ecell_beam_duration = ecell_durations['beam']
    ecell_ecell_duration = ecell_durations['ecell']
    if 'final_bg' in beam_n_rates:
        ecell_n_bg_df = pd.concat([ecell_n_rates['first_bg'],
                                   ecell_n_rates['final_bg']])
    else:
        ecell_n_bg_df = ecell_n_rates['first_bg']
    ecell_n_bg_rate = ecell_n_bg_df['Neutron rate (cps)'].mean()

    print(f"-----{beam_id}/{ecell_id}-----")
    print("---Counts---")
    print(f"Beam -  Background:   {beam_bg_count}")
    print(f"        Beam-loading: {beam_beam_count}")
    print(f"        Ecell active: {beam_ecell_count}")
    print(f"Ecell - Background:   {ecell_bg_count}")
    print(f"        Beam-loading: {ecell_beam_count}")
    print(f"        Ecell active: {ecell_ecell_count}")

    # print("---Durations---")
    # print(f"Background:   {bg_duration}")
    # print(f"Beam Loading: {beam_duration}")
    # print(f"Ecell:        {ecell_duration}")

    beam_bg_per_min = beam_bg_count / beam_bg_duration
    ecell_bg_per_min = ecell_bg_count / ecell_bg_duration
    beam_beam_per_min = beam_beam_count / beam_beam_duration
    beam_ecell_per_min = beam_ecell_count / beam_ecell_duration
    ecell_beam_per_min = ecell_beam_count / ecell_beam_duration
    ecell_ecell_per_min = ecell_ecell_count / ecell_ecell_duration
    print("---Counts (per minute)---")
    print(f"Beam -  Background:   {beam_bg_per_min:.2f}")
    print(f"        Beam-loading: {beam_beam_per_min:.2f}")
    print(f"        Ecell active: {beam_ecell_per_min:.2f}")
    print(f"Ecell - Background:   {ecell_bg_per_min:.2f}")
    print(f"        Beam-loading: {ecell_beam_per_min:.2f}")
    print(f"        Ecell active: {ecell_ecell_per_min:.2f}")

    print("---Background Neutron Rates---")
    print(f"Beam:  {beam_n_bg_rate:.2f} cps")
    print(f"Ecell: {ecell_n_bg_rate:.2f} cps")

    bg_delta = abs(ecell_bg_per_min - beam_bg_per_min)
    bg_delta_rel = bg_delta / min(beam_bg_per_min, ecell_bg_per_min)
    print("---Background Comparison---")
    print(f"Difference: {bg_delta:.2f}")
    print(f"Percent:    {bg_delta_rel:.2%}")

    beam_beam_no_bg = beam_beam_per_min - beam_bg_per_min
    beam_ecell_no_bg = beam_ecell_per_min - beam_bg_per_min
    ecell_beam_no_bg = ecell_beam_per_min - ecell_bg_per_min
    ecell_ecell_no_bg = ecell_ecell_per_min - ecell_bg_per_min
    print("---Relative to Background---")
    print(f"Beam -  Beam-loading: {beam_beam_no_bg:.2f}")
    print(f"        Ecell active: {beam_ecell_no_bg:.2f}")
    print(f"Ecell - Beam-loading: {ecell_beam_no_bg:.2f}")
    print(f"        Ecell active: {ecell_ecell_no_bg:.2f}")

    beam_percent_no_bg = (ecell_beam_no_bg - beam_beam_no_bg) / beam_beam_no_bg
    beam_percent_bg = (ecell_beam_per_min - beam_beam_per_min) / beam_beam_per_min
    ecell_percent_no_bg = (ecell_ecell_no_bg - beam_ecell_no_bg) / beam_ecell_no_bg
    ecell_percent_bg = (ecell_ecell_per_min - beam_ecell_per_min) / beam_ecell_per_min
    print("---Percent Increase---")
    print(f"Beam-loading - BG removed:  {beam_percent_no_bg:.2%}")
    print(f"               BG included: {beam_percent_bg:.2%}")
    print(f"Ecell active - BG removed:  {ecell_percent_no_bg:.2%}")
    print(f"               BG included: {ecell_percent_bg:.2%}")